In [49]:
import pandas as pd 
import numpy as np 

# Data Extraction / Loading

In [50]:
raw_online_retail=pd.read_csv("../data/raw/online_retail.csv")
raw_public_holidays = pd.read_csv("../data/external/publicHolidays.csv")
print(raw_online_retail.columns)
print(raw_public_holidays.columns)

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')
Index(['Unnamed: 0', 'countryOrRegion', 'holidayName', 'normalizeHolidayName',
       'isPaidTimeOff', 'countryRegionCode', 'date'],
      dtype='object')


# Section 2: Data Cleaning & Preprocessing

# 2.1 Cleaning Online Retail Dataset

In [ ]:
clean_online_retail = raw_online_retail.copy()
clean_public_holidays = raw_public_holidays.copy()

Customer Purchasing Behavior

Goal: Understand repeat purchases, purchase frequency, and customer spend.  

- Key Steps:

  - Removed rows with missing `CustomerID`.  

  - Removed purchases with `Quantity <= 0` or `UnitPrice <= 0`.  

  - Converted `InvoiceDate` to datetime.  

- Outcome: Dataset ready to calculate metrics like total spend per customer, average purchase frequency, and repeat purchase rate.


In [52]:
clean_online_retail['CustomerID'].isnull().sum()


np.int64(135080)

In [53]:
clean_online_retail=clean_online_retail.dropna(subset=['CustomerID'])
clean_online_retail['CustomerID'].isnull().sum()

np.int64(0)

b.check for duplicate records aka purchases with InvoiceNo
* StockCode
* Quantity
* UnitPrice 
* remove duplicate records

In [54]:
clean_online_retail.duplicated(subset=['InvoiceNo','StockCode','Quantity','UnitPrice'])
clean_online_retail=clean_online_retail.drop_duplicates(subset=['InvoiceNo','StockCode','Quantity','UnitPrice'])


split the order record data into 3 different datasets 
* purchased_dataset ->Quantity>0
* invalid_purchase_dataset= Quantity==0
* return_dataset ->Quantity<0

In [55]:
purchased_dataset =clean_online_retail[clean_online_retail['Quantity']>0].copy()
invalid_purchase_dataset=clean_online_retail[clean_online_retail['Quantity']==0].copy()
return_dataset=clean_online_retail[clean_online_retail['Quantity']<0].copy()

In [56]:
clean_online_retail.dtypes

InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object

DATA CLEANING IN PURCHASED_DATASET

2.Convert features to appropriate datatypes


In [57]:
purchased_dataset['InvoiceDate']=pd.to_datetime(purchased_dataset['InvoiceDate'])

Product Performance
- Goal: Identify top-selling and revenue-generating products.  

- Key Steps:
  - Ensured all `StockCode` values exist. 

  - Standardized `Description` (stripped whitespace).  

  - Quantity: Extreme high quantities capped at upperbound_q

UnitPrice: Rows with zero, negative, or extremely high prices removed (or filtered)

- Outcome: Prepared data for Top-N product analysis and product-level trend insights.

In [58]:

purchased_dataset=purchased_dataset[purchased_dataset['StockCode'].notnull()]



2.standardizing categorical/text data.
 * strip edges
 * convert to lowercase
 * remove special charecters

In [59]:
purchased_dataset_stand_cat=purchased_dataset.copy()
purchased_dataset_stand_cat['Description']=purchased_dataset_stand_cat['Description'].str.strip().str.lower().str.replace(r'[^a-z0-9 ]', '', regex=True)


3.Identify outliers of Quantity

Quantity Outliers(Only Quanity is checked because In a typical retail/customer purchasing dataset like the one you’re working on, Quantity is often the main numeric column that can have extreme or suspicious values.)

**Definition:**  
Quantity outliers are unusually large or small order quantities that deviate significantly from the typical purchase behavior. These extreme values can skew key metrics like average sales per order.

I**mpact on Analysis:**  
- Extremely large orders can inflate averages and give a misleading view of customer behavior.  
- Extremely small or negative quantities (e.g., returns) can distort insights.



Detect outlier is Quantity using IQR method

In [60]:

Q1_q=purchased_dataset_stand_cat['Quantity'].quantile(0.25)
Q3_q=purchased_dataset_stand_cat['Quantity'].quantile(0.75)
IQR_q=Q3_q-Q1_q

upperbound_q=Q3_q+1.5*IQR_q

# vals >lb=Q1-1.5*1qr and < Q3+1.5*1qr
quantity_outliers=purchased_dataset_stand_cat[(purchased_dataset_stand_cat['Quantity']>upperbound_q)]
print(quantity_outliers)

       InvoiceNo StockCode                       Description  Quantity  \
9         536367     84879     assorted colour bird ornament        32   
31        536370     10002        inflatable political globe        48   
44        536370     22492            mini paint set vintage        36   
46        536371     22086     paper chain kit 50s christmas        80   
65        536374     21258        victorian sewing box large        32   
...          ...       ...                               ...       ...   
541835    581579     23581            jumbo bag paisley park        40   
541865    581583     20725           lunch bag red retrospot        40   
541866    581583     85038    6 chocolate love heart tlights        36   
541867    581584     20832  red flock love heart photo frame        72   
541868    581584     85038    6 chocolate love heart tlights        48   

               InvoiceDate  UnitPrice  CustomerID         Country  
9      2010-12-01 08:34:00       1.69     1

Handling Outliers in Quantity


Extreme High Quantities (Bulk or Mistakes):

Extremely high quantities can skew metrics like total spent, frequency, and repeat purchases. Cap them using the IQR method to keep the analysis realistic.

In [61]:
purchased_df = purchased_dataset_stand_cat.copy()
purchased_df['Quantity'] = purchased_df['Quantity'].clip(
    upper=upperbound_q
)

In [62]:
purchased_df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

Detecting Outliers in Unitprice


In [63]:
Q1_u=purchased_df['UnitPrice'].quantile(0.25)
Q3_u=purchased_df['UnitPrice'].quantile(0.75)
IQR_u=Q3_u-Q1_u
lowerbound_u=Q1_u-1.5*IQR_u
upperbound_u=Q3_u+1.5*IQR_u
Unitprice_outliers_lowerbound=purchased_df[purchased_df['UnitPrice']>lowerbound_u]
Unitprice_outliers_upperbound=purchased_df[purchased_df['UnitPrice']<upperbound_u]



Handling Outliers in Unitprice


In [64]:
#negative outliers and extream large positive outliers of UnitPrice
new_purchased_dataset=purchased_df.copy()
new_purchased_dataset=purchased_df[(purchased_df['UnitPrice']>0)&(purchased_df['UnitPrice']<upperbound_u)].copy()


In [65]:
new_purchased_dataset.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

Revenue Drivers
- Goal: Determine main contributors to revenue.  
- Key Steps:  
  - Removed rows with invalid `Quantity` or `UnitPrice`.  

  - Created `Revenue = Quantity × UnitPrice`.  

  - Inspected revenue distribution for extreme values.  
  
- **Outcome:** Ready to identify top customers, top products, and main revenue contributors.


In [66]:
new_purchased_dataset['Revenue']=new_purchased_dataset['Quantity']*new_purchased_dataset['UnitPrice']

In [67]:
new_purchased_dataset['Revenue'].describe()

count    357993.000000
mean         14.600212
std          16.364582
min           0.001000
25%           4.200000
50%          10.500000
75%          17.700000
max         201.150000
Name: Revenue, dtype: float64

In [68]:
neg_revenue = new_purchased_dataset[new_purchased_dataset['Revenue'] < 0]
print(neg_revenue.shape)
print(neg_revenue.head())
zero_revenue = new_purchased_dataset[new_purchased_dataset['Revenue'] == 0]
print(zero_revenue.shape)
print(zero_revenue.head())

(0, 9)
Empty DataFrame
Columns: [InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country, Revenue]
Index: []
(0, 9)
Empty DataFrame
Columns: [InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country, Revenue]
Index: []


Customer Retention Patterns
- Goal:Measure repeat purchase behavior and loyalty.  
- Key Steps:
  -  Already Dropped rows without `CustomerID`.  
  - Sorted by `CustomerID` and `InvoiceDate`.  
  - Extracted Year and Month to create cohorts.  
- Outcome: Enables calculation of repeat purchase rate, time between purchases, and customer lifetime value.


In [69]:
new_purchased_dataset_sorted=new_purchased_dataset.sort_values(by=['CustomerID','InvoiceDate'])

In [70]:
new_purchased_dataset_sorted['Year']=new_purchased_dataset_sorted['InvoiceDate'].dt.year
new_purchased_dataset_sorted['Month']=new_purchased_dataset_sorted['InvoiceDate'].dt.month
new_purchased_dataset_sorted['day_of_week'] = new_purchased_dataset_sorted['InvoiceDate'].dt.dayofweek
new_purchased_dataset_sorted['week'] = new_purchased_dataset_sorted['InvoiceDate'].dt.isocalendar().week


#Find the first purchase date per customer
new_purchased_dataset_sorted['CohortMonth']=new_purchased_dataset_sorted.groupby('CustomerID')['InvoiceDate'].transform('min').dt.to_period('M')


In [71]:
clean_online_retail=new_purchased_dataset_sorted.copy()

# 2.2 External Holiday Dataset 

In [72]:
clean_public_holidays.columns=(clean_public_holidays.columns.str.lower().str.strip().str.replace(" ","_"))

In [73]:
clean_public_holidays.columns

Index(['unnamed:_0', 'countryorregion', 'holidayname', 'normalizeholidayname',
       'ispaidtimeoff', 'countryregioncode', 'date'],
      dtype='object')

In [74]:
clean_public_holidays=clean_public_holidays[['date','countryorregion','holidayname']]

In [75]:
req_public_holidays=clean_public_holidays.copy()
req_public_holidays['date']=pd.to_datetime(req_public_holidays['date'])

req_public_holidays=req_public_holidays.dropna(subset=['date','countryorregion'])
req_public_holidays=req_public_holidays.drop_duplicates(subset=['date','countryorregion','holidayname'])
req_public_holidays['day_of_week']=req_public_holidays['date'].dt.weekday
req_public_holidays['month']=req_public_holidays['date'].dt.month
req_public_holidays['is_weekend']=req_public_holidays['date'].dt.weekday>5
req_public_holidays['month_end']=req_public_holidays['date'].dt.is_month_end
req_public_holidays['month_start']=req_public_holidays['date'].dt.is_month_start

In [76]:
req_public_holidays['countryorregion'] = req_public_holidays['countryorregion'].astype(str).str.strip().str.title()
categorical_features = ['countryorregion']


In [77]:
req_public_holidays['holidayname']=req_public_holidays['holidayname'].str.strip().str.title()
req_public_holidays['isholiday']=(req_public_holidays['holidayname'].notna().astype(int))


In [78]:
req_public_holidays.columns
req_public_holidays.to_csv('../data/cleaned/holiday_dataset.csv',index=False)
clean_online_retail.to_csv('../data/cleaned/online_retail_dataset.csv',index=False)


# 2.3. Merging External Data into Cleaned Dataset

In [79]:

clean_online_retail=pd.merge(clean_online_retail,req_public_holidays,left_on=['InvoiceDate','Country'],right_on=['date','countryorregion'],how='left')


In [80]:
print(clean_online_retail)

       InvoiceNo StockCode                          Description  Quantity  \
0         541431     23166       medium ceramic top storage jar        27   
1         537626     85116       black candelabra tlight holder        12   
2         537626     22375    airline bag vintage jet set brown         4   
3         537626     71477      colour glass star tlight holder        12   
4         537626     22492               mini paint set vintage        27   
...          ...       ...                                  ...       ...   
357988    570715     22419                     lipstick pen red        12   
357989    570715     22866        hand warmer scotty dog design        12   
357990    573167     23264   set of 3 wooden sleigh decorations        27   
357991    573167     21824  painted metal star with holly bells        27   
357992    573167     21014         swiss chalet tree decoration        24   

               InvoiceDate  UnitPrice  CustomerID         Country  Revenue 

In [81]:
clean_online_retail= clean_online_retail.drop(
    columns=['day_of_week_y']
)

clean_online_retail = clean_online_retail.rename(
    columns={'day_of_week_x': 'day_of_week'}
)


In [82]:
clean_online_retail.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'Revenue', 'Year', 'Month',
       'day_of_week', 'week', 'CohortMonth', 'date', 'countryorregion',
       'holidayname', 'month', 'is_weekend', 'month_end', 'month_start',
       'isholiday'],
      dtype='object')

In [83]:
clean_online_retail=clean_online_retail.dropna(subset=['InvoiceNo',
    'StockCode',
    'Quantity',
    'UnitPrice',
    'InvoiceDate',
    'Revenue'])

In [84]:
clean_online_retail['CustomerID'].isnull().sum()

np.int64(0)

In [85]:
clean_online_retail['isholiday'].isnull().sum()

np.int64(357993)

In [86]:
clean_online_retail['isholiday']=(clean_online_retail['isholiday'].fillna(False) .infer_objects(copy=False))

C:\Users\meena\AppData\Local\Temp\ipykernel_3576\2470773520.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  clean_online_retail['isholiday']=(clean_online_retail['isholiday'].fillna(False) .infer_objects(copy=False))


In [87]:
clean_online_retail['holidayname'].isnull().sum()

np.int64(357993)

In [88]:
clean_online_retail['holidayname']=clean_online_retail['holidayname'].fillna("Non-holiday")

In [89]:
clean_online_retail['HolidayQuantity']=(clean_online_retail['Quantity']*clean_online_retail['isholiday'])

In [90]:
clean_online_retail['HolidayQuantity'].isnull().sum()

np.int64(0)

In [91]:
clean_online_retail.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'Revenue', 'Year', 'Month',
       'day_of_week', 'week', 'CohortMonth', 'date', 'countryorregion',
       'holidayname', 'month', 'is_weekend', 'month_end', 'month_start',
       'isholiday', 'HolidayQuantity'],
      dtype='object')

In [92]:
clean_online_retail[['Month', 'month']].drop_duplicates()

,Month,month
0,1,NaN
1,12,NaN
60,4,NaN
83,6,NaN
100,8,NaN
120,10,NaN
201,9,NaN
203,11,NaN
262,2,NaN
291,3,NaN


In [93]:
clean_online_retail.drop(columns=['month'])
clean_online_retail.rename(columns={'Month':'Month_num'},inplace=True)

In [94]:
clean_online_retail.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'Revenue', 'Year', 'Month_num',
       'day_of_week', 'week', 'CohortMonth', 'date', 'countryorregion',
       'holidayname', 'month', 'is_weekend', 'month_end', 'month_start',
       'isholiday', 'HolidayQuantity'],
      dtype='object')

In [112]:
clean_online_retail['YearMonth']=clean_online_retail['InvoiceDate'].dt.to_period('M')
clean_online_retail.to_csv("../data/cleaned/online_retail_cleaned.csv",index=False)
clean_online_retail.to_csv("online_retail_clean_backup.csv",index=False)

# Section 3: Feature Engineering & KPI Extraction

FEATURES FOR REGGRESSION
•	extracting features for Predicting next-month product sales,Forecast Revenue and Predict Future Demand

In [113]:
clean_online_retail=clean_online_retail.copy()
clean_online_retail['YearMonth']=clean_online_retail['InvoiceDate'].dt.to_period('M')

#Monthly aggregation of quantity per product 
#Monthly average of unit price per product
#Monthly holiday count
#monthly revenue per product
Monthly = (
    clean_online_retail.groupby(['StockCode', 'YearMonth']).agg(
        MonthlyQuantity=('Quantity', 'sum'),
        AvgUnitPrice=('UnitPrice', 'mean'),
        HolidayCount=('isholiday', 'sum'),
        MonthlyRevenue=('Quantity',lambda x:(x * clean_online_retail.loc[x.index,'UnitPrice']).sum())
    ).reset_index()
)


A lag is the value of a variable from a previous time period that is used as a predictor for the current or future time period.

It “looks back” in time.

Commonly used in time-series forecasting.

Helps capture patterns like trends, seasonality, and momentum.

Example in words

Lag_1 of February = January’s value

Lag_2 of March = January’s value

Lag_3 of April = January’s value

In [114]:
monthly1=Monthly.copy()
monthly1=monthly1.sort_values(['StockCode','YearMonth'])
for lag in [1,2,3]:
    monthly1[f'lag{lag}_quantity']=monthly1.groupby('StockCode')['MonthlyQuantity'].shift(lag)

A rolling mean quantity is the average sales of a product over the last k months, calculated for each month, to capture the trend.

In [115]:
monthly1['rolling_mean_3']=monthly1.groupby('StockCode')['MonthlyQuantity'].shift(1).rolling(window=3).mean()

In [116]:
monthly1['avg_unitprice']=(monthly1.groupby('StockCode')['AvgUnitPrice'].pct_change())

Growth Rate (Momentum)

 Month-over-month demand change

In [117]:
monthly1['GrowthRate']=monthly1.groupby('StockCode')['MonthlyQuantity'].pct_change()

Month (Seasonality)

In [118]:
monthly1['Month']=monthly1['YearMonth'].dt.month

Lag revenue of past 3 months
Revenue from a previous time period, shifted forward and used as a feature to predict the current or future period.

In [119]:
monthly1['LagRevenue1']=monthly1.groupby('StockCode')['MonthlyRevenue'].shift(1)
monthly1['LagRevenue2']=monthly1.groupby('StockCode')['MonthlyRevenue'].shift(2)
monthly1['LagRevenue3']=monthly1.groupby('StockCode')['MonthlyRevenue'].shift(3)


In [120]:
monthly1=monthly1.dropna()
monthly1.columns

Index(['StockCode', 'YearMonth', 'MonthlyQuantity', 'AvgUnitPrice',
       'HolidayCount', 'MonthlyRevenue', 'lag1_quantity', 'lag2_quantity',
       'lag3_quantity', 'rolling_mean_3', 'avg_unitprice', 'GrowthRate',
       'Month', 'LagRevenue1', 'LagRevenue2', 'LagRevenue3'],
      dtype='object')

In [121]:
monthly1.to_csv("../data/processed/regression_features.csv",index=False)
monthly1.to_csv("regression_features_backup.csv",index=False)


1. KPIs AND FEATURES  FOR PRODUCT-LEVEL CLASSIFICATION DATASET

TotalQuantitySold

Meaning: The total number of units sold over a specific period (day, month, quarter, year).

TotalRevenue
Meaning: The total money/revenue earned from selling a product over a period. 

AvgMonthlyQuantity
Meaning:The average number of units sold per month for a product.

2️⃣ AvgMonthlyRevenue

Meaning:
The average revenue earned per month for a product.

ActivemonthCount Definition:
The total number of months in which a product was sold (Quantity > 0).

StdMonthlyQuantity

Definition:
The standard deviation of monthly quantities sold for a product.
Measures how much monthly sales fluctuate around the average.


1️⃣ AvgUnitPrice

Meaning:
The average selling price of a product over a period (usually per month, per product, or overall). based on stockcode

PriceVariance

Meaning:
The variance (or fluctuation) of selling unitprice for a product (stockcode) over time.
Captures how much the price changes, which may affect revenue or demand


PeakMonthSales
Meaning:

Maximum number of units sold in a single month for a product.

Base Columns Needed:

StockCode, YearMonth, Quantity

Logic:

Aggregate monthly sales per product → take the maximum.



HolidaySalesRatio

Meaning:

Fraction of a product’s sales that occurred on holidays or special occasions.

Base Columns Needed:

Quantity, Date

HolidayFlag (1 if holiday, 0 otherwise)

Logic:

HolidaySalesRatio =
Quantity sold on holidays
Total Quantity sold
HolidaySalesRatio=
Total Quantity sold
Quantity sold on holidays
	​



In [122]:
clean_online_retail[['Month_num', 'YearMonth']].head()

,Month_num,YearMonth
0,1,2011-01
1,12,2010-12
2,12,2010-12
3,12,2010-12
4,12,2010-12


In [123]:
clean_online_retail[['InvoiceDate', 'YearMonth']].head()


,InvoiceDate,YearMonth
0,2011-01-18 10:01:00,2011-01
1,2010-12-07 14:57:00,2010-12
2,2010-12-07 14:57:00,2010-12
3,2010-12-07 14:57:00,2010-12
4,2010-12-07 14:57:00,2010-12


In [124]:
monthly_product=clean_online_retail.groupby(['StockCode','YearMonth']).agg(
     MonthlyQuantity=('Quantity','sum'),
     MonthlyRevenue=('Revenue','sum'),
     TotalHolidayQuantity=('HolidayQuantity','sum'),
     
     
).reset_index()

product_level=monthly_product.groupby('StockCode').agg(
    TotalQuantitySold=('MonthlyQuantity','sum'),
    TotalRevenue=('MonthlyRevenue','sum'),
    AvgMonthlyQuantity=('MonthlyQuantity','mean'),
    AvgMonthlyRevenue=('MonthlyRevenue','mean'),
    ActiveMonthCount=('YearMonth','nunique'),
    StdMonthlyQuantity=('MonthlyQuantity','std'),
    PeakMonthSale=('MonthlyQuantity','max'),
    TotalHolidayQuantity=('TotalHolidayQuantity','sum')
    
    ).reset_index()

product_level['HolidayRatio']=(product_level['TotalHolidayQuantity']/product_level['TotalQuantitySold'])

PricingFeatures=clean_online_retail.groupby('StockCode').agg(
    AvgUnitPrice=('UnitPrice','mean'),
    PriceVariance=('UnitPrice','var'),
   
).reset_index()

CustomerEngagement=clean_online_retail.groupby('StockCode').agg(
     UniqueCustomersCount=('CustomerID','nunique')
).reset_index()
    
product_classification_features=(product_level.merge(PricingFeatures,on='StockCode',how='left')
                                 .merge(CustomerEngagement,on='StockCode',how='left'))

product_classification_features['HolidayRatio']=product_classification_features['HolidayRatio'].fillna(0)



In [125]:
product_classification_features.columns

Index(['StockCode', 'TotalQuantitySold', 'TotalRevenue', 'AvgMonthlyQuantity',
       'AvgMonthlyRevenue', 'ActiveMonthCount', 'StdMonthlyQuantity',
       'PeakMonthSale', 'TotalHolidayQuantity', 'HolidayRatio', 'AvgUnitPrice',
       'PriceVariance', 'UniqueCustomersCount'],
      dtype='object')

In [126]:
product_classification_features.to_csv("../data/processed/product_classification_features.csv   ")

2. KPIs AND FEATURES FOR CUSTOMER CHURN CLASSIFICATION DATASET



TotalOrders:-Each InvoiceNo = one order

A customer may appear in many rows per invoice (multiple products)

🧠 Conceptual definition

Count of unique InvoiceNo per CustomerID

--------------------------------------

Recency (Days Since Last Purchase)
📌 What it means (business sense)

How recently a customer last purchased.

Low recency → customer is active

High recency → customer is at risk of churn

📊 What it means in data

Based on InvoiceDate

Uses the most recent purchase date of each customer

Number of days between the dataset’s reference date and the customer’s last InvoiceDate

------------------------------------------

AvgOrderValue (AOV)
📌 What it means (business sense)

How much money a customer spends per order on average.

Used to identify:

High-value customers

Premium buyers

📊 What it means in data

Order value = sum of (Quantity × UnitPrice) per invoice

🧠 Conceptual definition

Total Revenue of customer ÷ TotalOrders

--------------------------------------
ActiveMonths – practice-oriented explanation

Goal: Count how many distinct months each customer made purchases.

----------------------------------------

PurchaseFrequency – practice-oriented explanation

Goal: How frequently the customer orders relative to their activity months.

Step 1: Understand formula

PurchaseFrequency = TotalOrders ÷ ActiveMonths

In [127]:
clean_online_retail=clean_online_retail.copy()
TotalOrdersDS=clean_online_retail.groupby('CustomerID')['InvoiceNo'].nunique().reset_index(name='TotalOrders')
reference_date=clean_online_retail['InvoiceDate'].max()
recencyDS=clean_online_retail.groupby('CustomerID')['InvoiceDate'].max().reset_index()
recencyDS['Recency']=(reference_date - recencyDS['InvoiceDate']).dt.days
clean_online_retail['OrderValue']=clean_online_retail['Quantity']*clean_online_retail['UnitPrice']
order_revenue=clean_online_retail.groupby(['InvoiceDate','CustomerID'])['OrderValue'].sum().reset_index()
avg_order_value = order_revenue.groupby('CustomerID')['OrderValue'].mean().reset_index(name='AvgOrderValue')



clean_online_retail['YearMonth']=clean_online_retail['InvoiceDate'].dt.to_period('M')
grouped=clean_online_retail.groupby('CustomerID')
activemonthDS=grouped['YearMonth'].nunique().reset_index(name='ActiveMonth')

customer_featuresDS=TotalOrdersDS.merge(activemonthDS,on='CustomerID').merge(avg_order_value,on='CustomerID').merge(recencyDS,on='CustomerID')
customer_featuresDS['PurchaseFrequency']=(customer_featuresDS['TotalOrders']/customer_featuresDS['ActiveMonth'])

In [128]:
customer_featuresDS.isnull().sum()


CustomerID           0
TotalOrders          0
ActiveMonth          0
AvgOrderValue        0
InvoiceDate          0
Recency              0
PurchaseFrequency    0
dtype: int64

In [129]:
customer_featuresDS.duplicated().sum()

np.int64(0)

In [130]:
customer_featuresDS.columns

Index(['CustomerID', 'TotalOrders', 'ActiveMonth', 'AvgOrderValue',
       'InvoiceDate', 'Recency', 'PurchaseFrequency'],
      dtype='object')

Cap = limit extreme values so they don’t distort your analysis or model


In [131]:
for i in ['AvgOrderValue','PurchaseFrequency','Recency']:
    upper=customer_featuresDS[i].quantile(0.99)
    lower=customer_featuresDS[i].quantile(0.01)
    customer_featuresDS[i]=customer_featuresDS[i].clip(upper=upper,lower=lower)

In [132]:
customer_featuresDS.to_csv('../data/processed/customer_churn_features.csv ',index=False)